# Task 2A: Hyperparameter Experiments

Greedy search: change one parameter at a time, carry the best forward.

**Baseline**: fixed chunking, 1024 tokens, overlap 200, top_k=5, alpha=0.5, no reranking, e5-large

**Experiments**:
1. Chunk size: 512 vs 1024 vs 2048
2. Chunk overlap: 100 vs 200 vs 400
3. Chunking strategy: fixed vs recursive vs layout_aware
4. Top-K: 3 vs 5 vs 10
5. Alpha (hybrid weight): 0.3 vs 0.5 vs 0.7 vs 1.0
6. Reranking: on vs off

In [ ]:
import sys
sys.path.insert(0, "..")

import json
from tqdm import tqdm
from src.parsing import parse_all_pdfs
from src.pipeline import RAGPipeline
from src.evaluation import load_golden_dataset, run_experiment, results_to_dataframe
from src.config import DEFAULT_CONFIG

In [ ]:
parsed_texts = parse_all_pdfs(use_cache=True)
golden = load_golden_dataset("../data/golden_dataset.json")
print(f"Loaded {len(golden)} golden QA pairs")

In [ ]:
all_results = []

def run_exp(name: str, config_overrides: dict, parsed_texts=parsed_texts, golden=golden):
    """Helper: build pipeline, ingest, evaluate, store result."""
    config = {**DEFAULT_CONFIG, **config_overrides}
    config["collection_name"] = f"exp_{name.replace(' ', '_').lower()}"
    pipeline = RAGPipeline(config)
    pipeline.ingest(parsed_texts)
    result = run_experiment(pipeline, golden, name)
    all_results.append(result)
    print(f"\n{name}:")
    for metric in ["faithfulness", "answer_relevancy", "context_recall", "context_precision"]:
        print(f"  {metric}: {result.get(metric, 'N/A'):.4f}")
    return result

## Experiment 0: Baseline

In [ ]:
baseline = run_exp("Baseline", {
    "chunking_strategy": "fixed",
    "chunk_size": 1024,
    "chunk_overlap": 200,
    "top_k": 5,
    "alpha": 0.5,
    "use_reranking": False,
})

## Experiment 1: Chunk Size

In [ ]:
for size in [512, 2048]:
    run_exp(f"Chunk size {size}", {
        "chunk_size": size,
        "chunk_overlap": 200,
        "chunking_strategy": "fixed",
        "top_k": 5,
        "alpha": 0.5,
        "use_reranking": False,
    })

## Experiment 2: Chunk Overlap
Using best chunk_size from Experiment 1.

In [ ]:
# NOTE: Update best_chunk_size based on Experiment 1 results
best_chunk_size = 1024  # Update after running Experiment 1

for overlap in [100, 400]:
    run_exp(f"Overlap {overlap}", {
        "chunk_size": best_chunk_size,
        "chunk_overlap": overlap,
        "chunking_strategy": "fixed",
        "top_k": 5,
        "alpha": 0.5,
        "use_reranking": False,
    })

## Experiment 3: Chunking Strategy
Using best chunk_size and overlap from above.

In [ ]:
best_overlap = 200  # Update after running Experiment 2

for strategy in ["recursive", "layout_aware"]:
    run_exp(f"Strategy {strategy}", {
        "chunk_size": best_chunk_size,
        "chunk_overlap": best_overlap,
        "chunking_strategy": strategy,
        "top_k": 5,
        "alpha": 0.5,
        "use_reranking": False,
    })

## Experiment 4: Top-K

In [ ]:
best_strategy = "fixed"  # Update after running Experiment 3

for k in [3, 10]:
    run_exp(f"Top-K {k}", {
        "chunk_size": best_chunk_size,
        "chunk_overlap": best_overlap,
        "chunking_strategy": best_strategy,
        "top_k": k,
        "alpha": 0.5,
        "use_reranking": False,
    })

## Experiment 5: Alpha (Hybrid Weight)

In [ ]:
best_top_k = 5  # Update after running Experiment 4

for alpha in [0.3, 0.7, 1.0]:
    run_exp(f"Alpha {alpha}", {
        "chunk_size": best_chunk_size,
        "chunk_overlap": best_overlap,
        "chunking_strategy": best_strategy,
        "top_k": best_top_k,
        "alpha": alpha,
        "use_reranking": False,
    })

## Experiment 6: Reranking

In [ ]:
best_alpha = 0.5  # Update after running Experiment 5

run_exp("With Reranking", {
    "chunk_size": best_chunk_size,
    "chunk_overlap": best_overlap,
    "chunking_strategy": best_strategy,
    "top_k": best_top_k,
    "alpha": best_alpha,
    "use_reranking": True,
})

## Summary Table

In [ ]:
df = results_to_dataframe(all_results)
print(df.to_string(index=False))
df

In [ ]:
# Save results for Task 2B
import pickle
with open("../data/experiment_results.pkl", "wb") as f:
    pickle.dump(all_results, f)
print("Results saved to data/experiment_results.pkl")